In [ ]:
from datasets import load_dataset
import sys
sys.path.append("??/int/")
from grader import compute_score

### Load guidance

In [ ]:
dataset = load_dataset("CMU-AIRe/InT-hard-set-with-incorrect-attempts-and-interventions", split="train")

14766

In [ ]:
import pickle
import os
from tqdm import tqdm

results_dir = "OUTPUT_PATH"
all_results = {}

for start in tqdm(range(0, len(dataset), 256)):
    end = min(start + 256, len(dataset))
    pkl_name = f"guided_pass_at_32_{start}_{end}_train.pkl"
    if os.path.exists(os.path.join(results_dir, pkl_name)):
        pkl_path = os.path.join(results_dir, pkl_name)
        with open(pkl_path, "rb") as f:
            part = pickle.load(f)
        all_results.update(part)
    else:
        print(f"Skipping {start // 256} because it does not exist.")

print(f"Loaded {len(all_results)} results.")

  0%|          | 0/29 [00:00<?, ?it/s]

INFO 11-12 17:00:55 [__init__.py:239] Automatically detected platform cuda.


100%|██████████| 29/29 [01:47<00:00,  3.71s/it]

Loaded 14766 results.


In [ ]:
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed # <--- CHANGED
import sys
# NOTE: No more threading or locks needed!

# --- Assume dataset, all_results, and compute_score are defined ---
# (compute_score is a separate function, as you have it)

def process_index(i):
    """
    Processes a single index and RETURNS its result.
    It no longer accesses any global variables.
    """
    try:
        if i not in all_results:
            return None  # Return nothing to indicate no results
        
        local_scores = []
        
        for result in all_results[i].outputs:
            text = result.text
            answer = dataset[i]['answer']
            
            # This is now process-safe!
            score = compute_score(text, answer) 
            local_scores.append(score)
        
        # Return the index and the scores you found
        if local_scores:
            return (i, local_scores)
        return None

    except Exception as e:
        # This will catch errors *within* the process
        print(f"Error processing index {i}: {e}", file=sys.stderr)
        return None

# --- Main execution ---
scores = {} # This is now built in the main thread

# Use ProcessPoolExecutor instead of ThreadPoolExecutor
with ProcessPoolExecutor(max_workers=16) as executor: # Note: max_workers should be ~
                                                     # number of CPU cores, not 100
    
    futures = [executor.submit(process_index, i) for i in range(len(dataset))]
    
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            # Get the return value from the function
            result = future.result() 
            
            if result:
                # Unpack the (index, scores) tuple and add to dict
                i, local_scores = result
                scores[i] = local_scores
                
        except Exception as e:
            # This catches infrastructure errors
            print(f"A future raised an unexpected error: {e}", file=sys.stderr)

print(f"Completed processing {len(scores)} indices.")

In [5]:
import numpy as np
num_nonzero_scores = 0
total = 0
for i in scores:
    num_nonzero_scores += np.count_nonzero(scores[i])
    total += len(scores[i])

print(f"Number of nonzero scores: {num_nonzero_scores} / {total} ({num_nonzero_scores / total})")

Number of nonzero scores: 20170 / 472320 (0.04270409891598916)


In [6]:
oracle_leaked = 0
for i in scores:
    for j in range(32):
        if scores[i][j] == 1 and str(dataset[i]['answer']) in dataset[i]['intervention']['content']:
            oracle_leaked += 1

print(f"Oracle leaked: {oracle_leaked} / {num_nonzero_scores} ({oracle_leaked / num_nonzero_scores})")

Oracle leaked: 0 / 20170 (0.0)


In [ ]:
distinct_problems = set()
correct_indices = set()


for i in range(len(dataset)):
    if i not in scores:
        continue
    for j in range(32):
        if scores[i][j] == 1:
            distinct_problems.add(dataset[i]['problem'])
            correct_indices.add(i)
print(f"Number of distinct problems: {len(distinct_problems)} / {len(set(dataset['problem']))} ({len(distinct_problems) / len(set(dataset['problem']))})")

Number of distinct problems: 1076 / 4048 (0.2658102766798419)


### Filter dataset for interventions that give nonzero pass@32

In [ ]:
positive_dataset = dataset.filter(lambda x, i: i in correct_indices, with_indices=True)
positive_dataset.push_to_hub("CMU-AIRe/InT-SFT", split="train")

Filter:   0%|          | 0/14766 [00:00<?, ? examples/s]

Filter:   0%|          | 0/14766 [00:00<?, ? examples/s]

2213
12553
